In [1]:
import pandas as pd
import os
import numpy as np
# 1. 讀取原始資料
raw_data_path = '../data/raw/YRBS_2007 .csv'
df = pd.read_csv(raw_data_path)

# WhatIsYourSex: 性別 (1=Female, 2=Male)
# CurrentCigaretteUse / EverCigaretteUse: 過去30天抽菸天數 / 終身是否吸菸
# CurrentAlcoholUse / EverAlcoholUse: 過去30天飲酒天數 / 終身是否飲酒
# CurrentMarijuaUse / EverMarijuaUse: 過去30天大麻次數 / 終身是否吸食大麻
target_cols = [
    'WhatIsYourSex', 
    'CurrentCigaretteUse', 'EverCigaretteUse',
    'CurrentAlcoholUse', 'EverAlcoholUse',
    'CurrentMarijuaUse', 'EverMarijuaUse'
]
df_subset = df[target_cols].copy()

# 3. 刪除任何含有缺失值 (NaN) 的樣本
# 這能確保主要推論與未來延伸分析使用的是完全一致且乾淨的有效回答基底
df_clean = df_subset.dropna().copy()
print(f"原始資料總筆數: {len(df)}")
print(f"刪除任一欄位缺失值後筆數: {len(df_clean)}")

print("=" * 60)
print("🚀 正在執行精準變數重編碼 (Recoding)...")
print("=" * 60)

# 香菸與酒精的標準定義：codes 2-7 代表過去30天有使用(1)，code 1 代表完全沒有使用(0)
def recode_cigarette_alcohol(x):
    if x >= 2 and x <= 7:
        return 1  # 過去30天【有使用】
    elif x == 1:
        return 0  # 過去30天【沒使用】
    return np.nan

# 💡 根據資料稽核發現：大麻最大值為 6，故精準定義 codes 2-6 代表有使用(1)
def recode_marijuana(x):
    if x >= 2 and x <= 6:
        return 1  # 過去30天【有吸食大麻】
    elif x == 1:
        return 0  # 過去30天【沒吸食大麻】
    return np.nan

# 精準套用各自的過去 30 天重編碼函數
df_clean['Smoking_Status'] = df_clean['CurrentCigaretteUse'].apply(recode_cigarette_alcohol)
df_clean['Alcohol_Status'] = df_clean['CurrentAlcoholUse'].apply(recode_cigarette_alcohol)
df_clean['Marijuana_Status'] = df_clean['CurrentMarijuaUse'].apply(recode_marijuana)


def recode_ever_smoke(x):
    if x == 1: return 1     # 有吸過
    if x == 2: return 0     # 從未吸過
    return np.nan

def recode_ever_alc_mar(x):
    if x >= 2 and x <= 7: return 1  # 2~7歲數級距 ➔ 代表【有過經驗】
    if x == 1: return 0            # 1 ➔ 代表【從未嘗試】
    return np.nan

df_clean['Ever_Smoke_Recoded'] = df_clean['EverCigaretteUse'].apply(recode_ever_smoke)
df_clean['Ever_Alcohol_Recoded'] = df_clean['EverAlcoholUse'].apply(recode_ever_alc_mar)
df_clean['Ever_Marijuana_Recoded'] = df_clean['EverMarijuaUse'].apply(recode_ever_alc_mar)

df_final = df_clean.copy()

print("\n--- 開始執行修正後的數據邏輯一致性檢查 ---")

# (1) 抽菸邏輯檢查 (終身從未=0，但過去30天有抽=1 ➔ 矛盾)
smoke_conflict = (df_final['Ever_Smoke_Recoded'] == 0) & (df_final['Smoking_Status'] == 1)
print(f"👉 因【抽菸前後矛盾】被剔除的無效樣本: {smoke_conflict.sum()} 筆")
df_final = df_final[~smoke_conflict]

# (2) 飲酒邏輯檢查 (終身從未=0，但過去30天有喝=1 ➔ 矛盾)
alcohol_conflict = (df_final['Ever_Alcohol_Recoded'] == 0) & (df_final['Alcohol_Status'] == 1)
print(f"👉 因【飲酒前後矛盾】被剔除的無效樣本: {alcohol_conflict.sum()} 筆")
df_final = df_final[~alcohol_conflict]

# (3) 大麻邏輯檢查 (終身從未=0，但過去30天有吸=1 ➔ 矛盾)
marijuana_conflict = (df_final['Ever_Marijuana_Recoded'] == 0) & (df_final['Marijuana_Status'] == 1)
print(f"👉 因【大麻前後矛盾】被剔除的無效樣本: {marijuana_conflict.sum()} 筆")
df_final = df_final[~marijuana_conflict]

print(f"\n🎉 排除所有邏輯矛盾樣本後，最終有效母體樣本數: {len(df_final)}")

summary = df_final.groupby('WhatIsYourSex')['Smoking_Status'].value_counts(normalize=True).unstack()
print("\n" + "=" * 50)
print("--- 男女抽菸比例預覽 (0=不抽, 1=抽菸) ---")
print(summary)

print("\n--- 💡 延伸研究：多重成癮物質關聯性預覽 ---")
summary_alcohol = df_final.groupby('Smoking_Status')['Alcohol_Status'].value_counts(normalize=True).unstack()
print("\n[菸 vs 酒] 抽菸狀態與過去30天飲酒比例對比 (列代表菸，欄代表酒)：")
print(summary_alcohol)

summary_marijuana = df_final.groupby('Smoking_Status')['Marijuana_Status'].value_counts(normalize=True).unstack()
print("\n[菸 vs 大麻] 抽菸狀態與過去30天大麻比例對比 (列代表菸，欄代表大麻)：")
print(summary_marijuana)

from pathlib import Path

# 1. 定義輸出資料夾路徑（採用更穩健的 pathlib，避免跨作業系統路徑出錯）
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # 如果 processed 資料夾不存在，會自動建立
OUTPUT_FILE = OUTPUT_DIR / 'yrbs_smoking_cleaned.csv'

# 2. 嚴格篩選出分析所需的核心欄位（避免把 Ever_Recoded 等中間檢查用的變數匯出，保持格式乾淨）
final_cols = ['WhatIsYourSex', 'Smoking_Status', 'Alcohol_Status', 'Marijuana_Status']

# 3. 正式匯出為 CSV 檔案
df_final[final_cols].to_csv(OUTPUT_FILE, index=False, encoding='utf-8')


原始資料總筆數: 14041
刪除任一欄位缺失值後筆數: 11301
🚀 正在執行精準變數重編碼 (Recoding)...

--- 開始執行修正後的數據邏輯一致性檢查 ---
👉 因【抽菸前後矛盾】被剔除的無效樣本: 0 筆
👉 因【飲酒前後矛盾】被剔除的無效樣本: 0 筆
👉 因【大麻前後矛盾】被剔除的無效樣本: 0 筆

🎉 排除所有邏輯矛盾樣本後，最終有效母體樣本數: 11301

--- 男女抽菸比例預覽 (0=不抽, 1=抽菸) ---
Smoking_Status         0         1
WhatIsYourSex                     
1.0             0.825765  0.174235
2.0             0.779843  0.220157

--- 💡 延伸研究：多重成癮物質關聯性預覽 ---

[菸 vs 酒] 抽菸狀態與過去30天飲酒比例對比 (列代表菸，欄代表酒)：
Alcohol_Status         0         1
Smoking_Status                    
0               0.657048  0.342952
1               0.126069  0.873931

[菸 vs 大麻] 抽菸狀態與過去30天大麻比例對比 (列代表菸，欄代表大麻)：
Marijuana_Status         0         1
Smoking_Status                      
0                 0.897357  0.102643
1                 0.410626  0.589374
